# GRPO 强化学习训练
## 运筹优化建模的 RLHF 训练

本 notebook 用于在完成 SFT 后，运行 GRPO 强化学习训练。

### 关键改进
1. **修复版奖励函数**: 移除关键词奖励，防止 "python" 刷分攻击
2. **多维度奖励**: 格式奖励 + 执行奖励 + 答案奖励
3. **两步训练法**: 先 SFT 打基础，再 GRPO 提升

In [ ]:
# 安装依赖
!pip install -q torch transformers accelerate bitsandbytes peft trl datasets pyomo highspy tensorboard scikit-learn

In [ ]:
import torch
import json
import re
import numpy as np
import os

from dataclasses import dataclass
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset as HFDataset

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

## 1. 配置

In [ ]:
# ===== 配置 =====
# SFT 模型路径（从 train_sft.ipynb 获得）
SFT_MODEL_PATH = "./outputs/sft/final"  # 修改为你的 SFT 模型路径

# 训练数据
TRAIN_DATA_PATH = "./data/processed/train.jsonl"

# 输出
OUTPUT_DIR = "./outputs/grpo"

# 训练参数
MAX_SAMPLES = 5000       # 最大训练样本数
NUM_EPOCHS = 2           # GRPO 训练轮数
NUM_GENERATIONS = 4       # 每个 prompt 生成的样本数
LEARNING_RATE = 1e-5      # 学习率
KL_COEF = 0.04           # KL 惩罚系数
MAX_LENGTH = 1536         # 最大序列长度

## 2. 奖励函数（修复版）

In [ ]:
@dataclass
class RewardResult:
    total_reward: float
    format_reward: float
    execution_reward: float
    answer_reward: float = 0.0
    process_reward: float = 0.0
    is_valid: bool = False
    error_message: str = ""


class RewardFunction:
    """
    修复版奖励函数
    
    关键修复：
    1. 移除关键词奖励（原来 "python" 出现就能拿分）
    2. 必须有完整的 Pyomo 代码结构才能得格式分
    3. 代码必须能执行才能得执行分
    """
    
    # 格式子项：只有完整的代码结构才能得分
    FORMAT_ITEMS = {
        'import_pyomo': (r'import\s+pyomo|from\s+pyomo', 0.05),
        'model_def': (r'(Model|model)\s*=', 0.05),
        'variables': (r'(Var|var)\s*\(', 0.05),
        'objective': (r'(Objective|expr)\s*\(', 0.05),
        'constraints': (r'(Constraint|constraintlist)\s*\(', 0.05),
        'solver': (r'\.solve|\.SolverFactory', 0.05),
    }
    
    def __init__(
        self,
        format_scale=1.0,
        exec_scale=2.0,
        answer_scale=3.0,
        process_scale=0.5,
    ):
        self.format_scale = format_scale
        self.exec_scale = exec_scale
        self.answer_scale = answer_scale
        self.process_scale = process_scale
    
    def _extract_code(self, text):
        """从响应中提取代码块"""
        patterns = [
            r'```python\n(.*?)```',
            r'```pyomo\n(.*?)```',
            r'```\n(.*?)```',
        ]
        for p in patterns:
            m = re.search(p, text, re.DOTALL)
            if m:
                code = m.group(1).strip()
                # 验证是有效的 Pyomo 代码
                if 'pyomo' in code.lower() or 'import' in code:
                    return code
        return None
    
    def compute_format_reward(self, text):
        """计算格式奖励：必须包含完整的代码结构"""
        code = self._extract_code(text)
        if code is None:
            return 0.0, {}
        
        reward = 0.0
        item_results = {}
        critical_passed = 0
        
        for name, (pattern, weight) in self.FORMAT_ITEMS.items():
            matched = bool(re.search(pattern, code, re.IGNORECASE))
            item_results[name] = matched
            if matched:
                reward += weight
                if name in ['import_pyomo', 'model_def', 'solver']:
                    critical_passed += 1
        
        # 完整性奖励：所有关键项都通过时额外加分
        if critical_passed == 3:
            reward += 0.15
        
        return min(reward * self.format_scale, 1.5), item_results
    
    def _execute_pyomo(self, code):
        """执行 Pyomo 代码"""
        import sys, io
        from contextlib import redirect_stdout, redirect_stderr
        
        exec_globals = {}
        try:
            from pyomo.environ import (
                ConcreteModel as CM, Objective as Obj, Constraint as C,
                Var as V, SolverFactory as SF, value as val,
                minimize as min_, maximize as max_,
            )
            exec_globals.update({
                'ConcreteModel': CM, 'Objective': Obj, 'Constraint': C,
                'Var': V, 'SolverFactory': SF, 'value': val,
                'minimize': min_, 'maximize': max_,
            })
        except ImportError as e:
            return False, None, "Pyomo 未安装"
        
        stdout = io.StringIO()
        try:
            with redirect_stdout(stdout), redirect_stderr(io.StringIO()):
                exec(code, exec_globals)
            return True, None, ""
        except Exception as e:
            return False, None, str(e)
    
    def compute_execution_reward(self, text):
        """计算执行奖励：代码必须能实际运行"""
        code = self._extract_code(text)
        if code is None:
            return 0.0, False, ""
        
        success, solved_val, error = self._execute_pyomo(code)
        if success:
            return 1.0 * self.exec_scale, True, ""
        return 0.0, False, error
    
    def compute_process_reward(self, text):
        """计算过程奖励：评估推理链质量"""
        reward = 0.0
        
        if re.search(r'##\s*(问题分析|Problem)', text):
            reward += 0.3
        if re.search(r'##\s*(数学模型|Mathematical)', text):
            reward += 0.3
        
        return min(reward * self.process_scale, 0.6)
    
    def __call__(self, text, ground_truth=None, is_last=False):
        # 1. 格式奖励
        format_reward, _ = self.compute_format_reward(text)
        
        # 2. 执行奖励
        exec_reward = 0.0
        is_valid = False
        error_msg = ""
        
        # 只有格式基本正确才尝试执行（防止无意义代码浪费计算）
        if is_last and format_reward >= 0.5:
            exec_reward, is_valid, error_msg = self.compute_execution_reward(text)
        
        # 3. 过程奖励
        process_reward = self.compute_process_reward(text)
        
        # 总奖励
        total = format_reward + exec_reward + process_reward
        total = max(min(total, 4.0), -1.0)  # 裁剪到合理范围
        
        return RewardResult(
            total_reward=total,
            format_reward=format_reward,
            execution_reward=exec_reward,
            answer_reward=0.0,
            process_reward=process_reward,
            is_valid=is_valid,
            error_message=error_msg,
        )

In [ ]:
# 测试奖励函数
print("测试奖励函数...")

rf = RewardFunction()

# 测试1: 刷分攻击（应该得0分）
hack_text = "python python python python python python python python"
r = rf(hack_text, is_last=True)
print(f"\n刷分攻击: total={r.total_reward:.3f}, format={r.format_reward:.3f}")
assert r.total_reward == 0.0, "应该完全抵御刷分攻击！"
print("  ✓ 成功抵御刷分攻击")

# 测试2: 有效 Pyomo 代码
valid_code = """
```python
from pyomo.environ import *
m = ConcreteModel()
m.x = Var(within=NonNegativeReals)
m.y = Var(within=NonNegativeReals)
m.obj = Objective(expr=3*m.x + 2*m.y, sense=maximize)
m.c1 = Constraint(expr=m.x + m.y <= 10)
m.c2 = Constraint(expr=2*m.x + m.y <= 16)
SolverFactory('cbc').solve(m)
```
"""
r = rf(valid_code, is_last=True)
print(f"\n有效代码: total={r.total_reward:.3f}, format={r.format_reward:.3f}, exec={r.execution_reward:.3f}")
print(f"  执行状态: {'成功' if r.is_valid else '失败'}")
print(f"  错误: {r.error_message or '无'}")

print("\n奖励函数测试通过！")

## 3. 准备数据和模型

In [ ]:
# 加载数据
print(f"加载训练数据: {TRAIN_DATA_PATH}")
train_data = []
with open(TRAIN_DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                train_data.append(json.loads(line))
            except:
                continue

if MAX_SAMPLES > 0 and len(train_data) > MAX_SAMPLES:
    import random
    random.seed(42)
    train_data = random.sample(train_data, MAX_SAMPLES)

print(f"训练样本数: {len(train_data)}")

In [ ]:
# 加载 tokenizer
print(f"加载 tokenizer from: {SFT_MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 加载 SFT 模型
print("加载 SFT 模型...")
model = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_PATH,
    torch_dtype=torch.float32,
    device_map='auto',
    trust_remote_code=True,
)

# 如果模型没有 LoRA 权重，应用 LoRA
if not hasattr(model, 'peft_type') or model.peft_type is None:
    print("应用 LoRA...")
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        bias='none',
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [ ]:
# 准备 HuggingFace Dataset
texts = []
answers = []

for item in train_data:
    prompt = item.get('prompt', '')
    response = item.get('response', '')
    
    # 构建对话格式
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response},
    ]
    
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False)
    except:
        # 如果 tokenizer 不支持 chat template，用简单格式
        text = f"{prompt}\n\n{response}"
    
    texts.append(text)
    answers.append(item.get('answer', ''))

dataset = HFDataset.from_dict({"text": texts, "answer": answers})
print(f"数据集大小: {len(dataset)}")

## 4. 运行 GRPO 训练

In [ ]:
# 创建奖励函数
reward_fn = RewardFunction()

# 包装奖励函数
def reward_function(prompts, responses, **kwargs):
    rewards = []
    for resp in responses:
        result = reward_fn(resp, is_last=True)
        rewards.append(result.total_reward)
    
    # 打印奖励分布
    if len(rewards) > 0:
        print(f"  [reward] mean={np.mean(rewards):.3f}, std={np.std(rewards):.3f}, "
              f"max={np.max(rewards):.3f}, min={np.min(rewards):.3f}")
    
    return rewards

In [ ]:
# GRPO 配置
grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_LENGTH // 2,
    max_completion_length=MAX_LENGTH // 2,
    num_generations=NUM_GENERATIONS,
    beta=KL_COEF,
    loss_type='grpo',
    fp16=torch.cuda.is_available(),
    report_to=['tensorboard'],
    seed=42,
)

In [ ]:
# 创建 GRPO 训练器
print("创建 GRPO 训练器...")
print(f"  模型: {SFT_MODEL_PATH}")
print(f"  样本数: {len(dataset)}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  每样本生成数: {NUM_GENERATIONS}")
print(f"  学习率: {LEARNING_RATE}")
print(f"  KL 系数: {KL_COEF}")

trainer = GRPOTrainer(
    model=model,
    config=grpo_config,
    reward_function=reward_function,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

In [ ]:
# 开始训练
print("="*60)
print("开始 GRPO 训练")
print("="*60)

trainer.train()

print("\n训练完成！保存模型...")
final_path = f"{OUTPUT_DIR}/final"
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)

print(f"模型已保存: {final_path}")

## 5. 查看 TensorBoard 日志

In [ ]:
# 在 Colab 中查看 TensorBoard
# %load_ext tensorboard
# %tensorboard --logdir=./outputs/grpo/logs/